# <span style="color:#F57C00">Modeling: Predicting the Summer Surface Temperature of a District</span>

**What the model does.** It gets the facts of a district (green share, buildings, elevation, population, region, climate) and the weather of the summer, and it says how hot the surface of the built-up area is (Landsat surface temperature, July + August, in °C). It learns this from the comparison of districts, mostly *inside the same city* (see the EDA notebook). The same model is used in two ways:
1. **Prediction:** the surface temperature of a district in a summer. We test it on years it has not seen.
2. **What-if:** the same district with a different green share, everything else unchanged: how much cooler or hotter does it get?

**How we test it fairly** (the EDA showed that a random split of the rows is far too optimistic, because the same district appears in many years and neighbours look alike):

| Data | Years | Used for |
| :--- | :--- | :--- |
| **Training** | 2013-2022 | Choosing the algorithm, its settings and the improvements, with two tests (below) that never let a model see what it is tested on |
| **Test** | 2023-2025 | One exam at the end (Part 6): every test year is predicted by a model trained on the *earlier* years only ("walk-forward") |
| **Final model** | 2013-2025 | Trained on everything after the exam, to be used for new predictions and what-if. It is **not scored again**: all scores come from models that did not see the year they were tested on, so this is no leakage. |

**Two tests, because there are two questions.** The model will serve the 829 districts that are in the data, but it should also be sound for places it has not seen:

| Test | Question | How |
| :--- | :--- | :--- |
| **New cities** | Does it work for a city that it has never seen? | 5-fold cross-validation that keeps **whole cities** together |
| **Next summers** | Does it predict the next summer of districts that it knows? | Train on the years before *t*, predict *t*, for *t* = 2019, 2020, 2021, 2022 |

Both are scored with the same number (Part 2); the **joint score** is their mean. Every algorithm gets exactly the same search (Part 3), so the choice (Part 4) is fair. Part 5 then improves the chosen model, with the same tests and a rule that a change must beat the old model with a margin that is not noise.

**The six parts**

| Part | What happens |
| :--- | :--- |
| **1. Data preparation** | Read the processed tables, join them on `district_code`, build the features, split into training and test years |
| **2. Scoring and the two tests** | How a model is scored, and the two tests as small functions |
| **3. The same deep search for every algorithm** | 6 algorithms and a linear baseline; Optuna runs `N_TRIALS` (30 by default) trials for each, 16 at the same time |
| **4. Choosing the model** | The best trials are tested again on new seeds; a rule that is written down before the result decides |
| **5. Improving the model** | Two more feature groups (clear sky, climate normals) and the air temperature as an offset of the target; tested on fresh seeds and with a bootstrap over cities |
| **6. Exam and final model** | Walk-forward exam on 2023-2025 against the model before Part 5, a what-if check, the feature importance, the final training on all years and saving (one file) |

**Run time.** The search is the long part: with the default `N_TRIALS = 30` it takes about 25-35 minutes on 16 CPU cores for all algorithms (the first full run used 150 trials each and took about 2 hours). Its trials are saved in `data/models/search/`, so a second run only reads them. The fresh-seed tests of Part 4 (about 15 minutes) and the probes of Part 5 (about 3 minutes) run every time. Set `N_TRIALS` to 10 to try the whole notebook in a few minutes.

# <span style="color:#F57C00">Part 1: Data Preparation</span>

We only use the processed tables of `data/processed` and join them in memory on `district_code`, like in the EDA notebook. One row of the model table is **one district and one summer** (July + August).

## Step 1: Libraries, settings and the tables

In [ ]:
import hashlib
import json
import warnings
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostRegressor
from joblib import Parallel, delayed
from lightgbm import LGBMRegressor
from scipy import stats
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
threadpool_limits(
    1
)  # every model uses ONE thread; the search runs many models at the same time instead (the data is small)

PROCESSED = Path("../data/processed")
MODELS = Path("../data/models")
SEARCH = (
    MODELS / "search"
)  # the trials of the search are saved here, so a second run does not repeat it
TARGET = "landsat_lst_c"  # the summer surface temperature of the built-up area (°C)
TRAIN_END = 2022  # last training year; 2023-2025 stay hidden until Part 6
N_TRIALS = 30  # trials of the search per algorithm in Part 3 (the first full run used 150)

green = pd.read_csv(PROCESSED / "greenness_districts_summer_2013_2025.csv")
climate = pd.read_csv(PROCESSED / "climate_districts_monthly_2000_2025.csv")
population = pd.read_csv(PROCESSED / "tuik_districts_population_clean.csv")
static = pd.read_csv(PROCESSED / "districts_urban_static.csv")
districts = gpd.read_file(PROCESSED / "turkey_districts_master.geojson")
centers = pd.DataFrame(
    districts[["district_code", "center_lat", "center_lon"]]
)  # only the center points, without the polygons
print(
    f"greenness {green.shape} | climate {climate.shape} | population {population.shape} | static {static.shape} | centers {centers.shape}"
)

## Step 2: Weather and population features per district and year
- **Weather of the summer** (July + August): mean temperature (`temp_mean_ja`) and number of hot days (`hot_days_ja`). **Rain of the spring** (March-June, `precip_spring`): a dry spring means less green in summer.
- **Population:** share of the people aged 65+ (`share65`) and the logarithm of the population (`logpop`, the size of the district).
These are the **urban** populations of TÜİK; we never divide an area by them (see the data README).

In [ ]:
summer = (
    climate[climate["month"].isin([7, 8])]
    .groupby(["district_code", "year"])
    .agg(temp_mean_ja=("temp_mean", "mean"), hot_days_ja=("hot_days", "sum"))
    .reset_index()
)
spring = (
    climate[climate["month"].between(3, 6)]
    .groupby(["district_code", "year"])
    .agg(precip_spring=("precip_sum", "sum"))
    .reset_index()
)
weather = summer.merge(
    spring, on=["district_code", "year"], validate="one_to_one"
)  # all districts and years (also used in Part 6)

age_65_plus = [
    c
    for c in population.columns
    if c.startswith("total_")
    and c.split("_")[1][:2] in ("65", "70", "75", "80", "85", "90")
]
population["share65"] = population[age_65_plus].sum(axis=1) / population["total_pop"]
population["logpop"] = np.log(population["total_pop"])
weather.head(3)

## Step 3: Join the tables
The greenness table is the base (one row per district and summer). `validate=` stops with an error if a join would repeat rows.

In [ ]:
table = (
    green.merge(
        weather, on=["district_code", "year"], how="left", validate="one_to_one"
    )
    .merge(
        population[["district_code", "year", "share65", "logpop"]],
        on=["district_code", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        static[["district_code", "elevation_m", "built_share_2020"]],
        on="district_code",
        how="left",
        validate="many_to_one",
    )
    .merge(centers, on="district_code", how="left", validate="many_to_one")
)
assert len(table) == len(green), "a join must not change the number of rows"
print(len(table), "district-summers")

## Step 4: Keep the reliable rows and build the features
- **Reliable rows** (EDA, parts A2 and C5): a built-up area of at least 0.5 km² and at least half of its pixels clear. Only cities with at least 3 such districts in a year stay, because we compare the districts *inside* a city.
- **Region:** the 7 geographic regions of Turkey as 0/1 columns (`pd.get_dummies`), so the model can learn that the Southeast is hotter and the Black Sea greener.
- **Within-city columns:** for the most important features we add the value *minus the mean of the same city in the same year* (`..._vs_city`). Weather and regional climate drop out, what is left is the difference between the districts of one city: the part that parks and buildings can change (EDA, parts C1 and C8).
`add_within_city` is a small function because we use it again in Parts 4, 5 and 6.

In [ ]:
REGIONS = {
    "Marmara": [10, 11, 16, 17, 22, 34, 39, 41, 54, 59, 77],
    "Aegean": [3, 9, 20, 35, 43, 45, 48, 64],
    "Mediterranean": [1, 7, 15, 31, 32, 33, 46, 80],
    "Central Anatolia": [6, 18, 26, 38, 40, 42, 50, 51, 58, 66, 68, 70, 71],
    "Black Sea": [5, 8, 14, 19, 28, 29, 37, 52, 53, 55, 57, 60, 61, 67, 69, 74, 78, 81],
    "Eastern Anatolia": [4, 12, 13, 23, 24, 25, 30, 36, 44, 49, 62, 65, 75, 76],
    "Southeastern Anatolia": [2, 21, 27, 47, 56, 63, 72, 73, 79],
}
region_of_city = {city: region for region, cities in REGIONS.items() for city in cities}

WEATHER = ["temp_mean_ja", "hot_days_ja", "precip_spring"]
WITHIN_CITY = [
    "landsat_green_share",
    "elevation_m",
    "built_share_2020",
    "log_urban_km2",
    "logpop",
    "share65",
    "temp_mean_ja",
    "hot_days_ja",
]


def add_within_city(frame, columns):
    # every value minus the mean of the same city in the same year
    frame = frame.copy()
    city_means = frame.groupby(["city_code", "year"])[columns].transform("mean")
    frame[[f"{name}_vs_city" for name in columns]] = (
        frame[columns] - city_means
    ).to_numpy()
    return frame


table["log_urban_km2"] = np.log(table["urban_km2"])
reliable = (
    (table["urban_km2"] >= 0.5)
    & (table["landsat_cover"] >= 0.5)
    & table[TARGET].notna()
)
table = table[reliable].dropna(
    subset=["share65", "logpop"]
)  # 3 districts that exist only later have no population in the early years
table = table[
    table.groupby(["city_code", "year"])["district_code"].transform("size") >= 3
]

region_columns = pd.get_dummies(
    table["city_code"].map(region_of_city), prefix="region", dtype=int
)[[f"region_{name}" for name in REGIONS]]
table = add_within_city(
    pd.concat([table, region_columns], axis=1), WITHIN_CITY
).reset_index(drop=True)

FEATURES = (
    [
        "landsat_green_share",
        "elevation_m",
        "built_share_2020",
        "log_urban_km2",
        "logpop",
        "share65",
        "temp_mean_ja",
        "hot_days_ja",
        "precip_spring",
        "center_lat",
        "center_lon",
    ]
    + list(region_columns.columns)
    + [f"{name}_vs_city" for name in WITHIN_CITY]
)
# More green must never mean a hotter surface: -1 = the prediction may only go down when the value goes up (the what-if depends on it)
MONOTONE = [
    -1 if name in ("landsat_green_share", "landsat_green_share_vs_city") else 0
    for name in FEATURES
]
print(
    f"{len(table)} rows | {table['district_code'].nunique()} districts | {table['city_code'].nunique()} cities | {len(FEATURES)} features"
)
table[FEATURES].isna().sum().sum()

## Step 5: Split into training and test years
**2013-2022 are the training years; 2023-2025 are the test years and are not touched until Part 6.** Everything in Parts 2 to 5 uses `train` only.

In [ ]:
train = table[table["year"] <= TRAIN_END].reset_index(drop=True)
test = table[table["year"] > TRAIN_END].reset_index(drop=True)
X_train, y_train = train[FEATURES], train[TARGET]

print(
    f"training: {len(train)} rows, years {train['year'].min()}-{train['year'].max()}, {train['city_code'].nunique()} cities"
)
print(f"test: {len(test)} rows, years {test['year'].min()}-{test['year'].max()}")
assert (
    train["year"].max() < test["year"].min()
), "the test years must come after the training years"

**What we learned (Part 1)**
- The model table has 10,622 rows: one summer of one of 829 districts in 80 cities, with 26 features. The training years (2013-2022) have 8,137 rows and the test years (2023-2025) 2,485 rows.
- The other 144 districts of Turkey have no row: 142 have a measured built-up area under 0.5 km² in every year, and the 2 reliable districts of Bayburt are left out because a city needs at least 3 reliable districts for the comparison inside the city.

# <span style="color:#F57C00">Part 2: Scoring and the Two Tests</span>

## Step 1: How a model is scored
We use three numbers (they are explained in the EDA, part C8):
- `rmse`: the error in °C over all rows;
- `r2_level`: how well the **level of a whole city** in a summer is predicted (compare the city-year means);
- `r2_within`: how well it is predicted **which districts of a city are hotter** (compare the values minus the city-year mean). This is the part the what-if is about.

`balance = r2_level + r2_within`: a model must be good at both, so we add them. This is the number that we maximise.

In [ ]:
def scores(rows, predicted):
    # rows: the districts that were predicted; predicted: the predictions in the same order
    result = rows[["city_code", "year", TARGET]].assign(predicted=np.asarray(predicted))
    city_year = result.groupby(["city_code", "year"])[[TARGET, "predicted"]]
    level = city_year.mean()  # one value per city and year
    inside = result[[TARGET, "predicted"]] - city_year.transform(
        "mean"
    )  # the part inside the cities
    r2_level = r2_score(level[TARGET], level["predicted"])
    r2_within = r2_score(
        inside[TARGET], inside["predicted"]
    )  # the mean of `inside` is 0, so this is 1 - error / variance inside the cities
    return pd.Series(
        {
            "rmse": root_mean_squared_error(result[TARGET], result["predicted"]),
            "r2_level": r2_level,
            "r2_within": r2_within,
            "balance": r2_level + r2_within,
        }
    )

## Step 2: The two tests as functions
Both tests give a prediction for every row they test, and a model never predicts what it has seen:
- **New cities:** `GroupKFold` with `shuffle=True` puts whole cities into 5 folds; each fold is predicted by a model trained on the other four. The score is the mean `balance` of the 5 folds.
- **Next summers:** for each of the summers 2019 to 2022 a model is trained on the years before it and predicts it. The score is the `balance` of the four summers together.

`make(seed)` is any function that returns a new, untrained model; this is how every algorithm of Part 3 uses the same two tests. The joint score is the mean of the two.

In [ ]:
FOLDS = 5
NEXT_YEARS = (2019, 2020, 2021, 2022)  # the summers that the second test predicts
next_rows = pd.concat([train[train["year"] == t] for t in NEXT_YEARS]).reset_index(
    drop=True
)  # in the order of the predictions


def predict_new_cities(make, seed, columns=None):
    # test 1: out-of-fold predictions; every city is predicted by a model that has not seen it
    columns = (
        columns or FEATURES
    )  # the columns the model gets (default: the features of Part 1)
    pred, fold = np.empty(len(train)), np.empty(len(train), dtype=int)
    splits = GroupKFold(n_splits=FOLDS, shuffle=True, random_state=seed).split(
        train, y_train, groups=train["city_code"]
    )
    for k, (fit_index, test_index) in enumerate(splits):
        pred[test_index] = (
            make(seed)
            .fit(train.iloc[fit_index][columns], y_train.iloc[fit_index])
            .predict(train.iloc[test_index][columns])
        )
        fold[test_index] = k
    return pred, fold


def score_new_cities(pred, fold):
    return np.mean(
        [scores(train[fold == k], pred[fold == k])["balance"] for k in range(FOLDS)]
    )


def predict_next_summers(make, seed, columns=None):
    # test 2: every summer is predicted by a model that was trained on the years before it
    columns = columns or FEATURES
    out = []
    for year in NEXT_YEARS:
        earlier, current = train[train["year"] < year], train[train["year"] == year]
        out.append(
            make(seed).fit(earlier[columns], earlier[TARGET]).predict(current[columns])
        )
    return np.concatenate(out)


def score_next_summers(pred):
    return scores(next_rows, pred)["balance"]


# a first look with the linear baseline: the two scores of a simple model
ridge = lambda seed: make_pipeline(StandardScaler(), Ridge(alpha=122))
new_cities = score_new_cities(*predict_new_cities(ridge, 100))
next_summers = score_next_summers(predict_next_summers(ridge, 100))
print(
    f"Ridge: new cities {new_cities:.3f} | next summers {next_summers:.3f} | joint {(new_cities + next_summers) / 2:.3f}"
)

**What we learned (Part 2)**
- A **new-city** test and a **next-summer** test ask different questions, and an algorithm can be better at one than at the other. The modeling of the earlier version of this notebook used only the first test; the app will serve known districts, so the second one matters as much.
- Both tests use training years only. The exam of Part 6 is still closed.

# <span style="color:#F57C00">Part 3: The Same Deep Search for Every Algorithm</span>

Six well-known algorithms and a linear baseline. **All of them get exactly the same treatment**: the same features, the same two tests, the same monotone rule for the green share, the same sampler and **`N_TRIALS` Optuna trials each** (30 by default). Nobody is tuned less than the others.

| Algorithm | Kind |
| :--- | :--- |
| Ridge | linear model (the baseline to beat) |
| RandomForest, ExtraTrees | many independent trees; ExtraTrees chooses the cut points at random, which makes it smoother |
| HistGradientBoosting, XGBoost, LightGBM, CatBoost | boosting: small trees that correct the errors of the earlier trees |

Green share is *monotone* (more green never means hotter) in every algorithm that can do it: the what-if depends on it. Ridge is a straight line and does not need it.

**How the search runs.** Optuna (TPE sampler) proposes settings, scores them with the two tests (`new cities` with two different splits, `next summers` once) and learns from the result. **16 trials run at the same time**, each model on **one** thread: the data is small, so this uses all cores far better than one model with 16 threads. The search spaces are wide and the same for everybody in spirit (trees, depth, learning rate, regularisation, sampling); only the extreme corners that are slow and never won (for example depth 10 with 1,500 trees in CatBoost) are capped.

## Step 1: The algorithms and their search spaces
`CANDIDATES[name](params, seed)` builds a model; `space(name, trial)` lists what Optuna may change.

In [ ]:
CANDIDATES = {
    "XGBoost": lambda p, seed: XGBRegressor(
        **p,
        monotone_constraints=tuple(MONOTONE),
        tree_method="hist",
        n_jobs=1,
        random_state=seed
    ),
    "LightGBM": lambda p, seed: LGBMRegressor(
        **p,
        subsample_freq=1,
        monotone_constraints=MONOTONE,
        n_jobs=1,
        random_state=seed,
        verbose=-1
    ),
    "CatBoost": lambda p, seed: CatBoostRegressor(
        **p,
        monotone_constraints=MONOTONE,
        thread_count=1,
        random_seed=seed,
        verbose=0,
        allow_writing_files=False
    ),
    "HistGradientBoosting": lambda p, seed: HistGradientBoostingRegressor(
        **p, monotonic_cst=MONOTONE, early_stopping=False, random_state=seed
    ),
    "RandomForest": lambda p, seed: RandomForestRegressor(
        **p, monotonic_cst=MONOTONE, n_jobs=1, random_state=seed
    ),
    "ExtraTrees": lambda p, seed: ExtraTreesRegressor(
        **p, monotonic_cst=MONOTONE, n_jobs=1, random_state=seed
    ),
    "Ridge": lambda p, seed: make_pipeline(StandardScaler(), Ridge(**p)),
}


def space(name, t):
    # t is an Optuna trial: t.suggest_int / suggest_float / suggest_categorical pick one value of the range
    if name == "XGBoost":
        return dict(
            n_estimators=t.suggest_int("n_estimators", 100, 1200, step=50),
            learning_rate=t.suggest_float("learning_rate", 0.005, 0.3, log=True),
            max_depth=t.suggest_int("max_depth", 2, 10),
            min_child_weight=t.suggest_float("min_child_weight", 1, 100, log=True),
            subsample=t.suggest_float("subsample", 0.4, 1.0),
            colsample_bytree=t.suggest_float("colsample_bytree", 0.2, 1.0),
            reg_lambda=t.suggest_float("reg_lambda", 0.01, 100, log=True),
            reg_alpha=t.suggest_float("reg_alpha", 0.001, 10, log=True),
            gamma=t.suggest_float("gamma", 0.001, 5, log=True),
        )
    if name == "LightGBM":
        return dict(
            n_estimators=t.suggest_int("n_estimators", 100, 1200, step=50),
            learning_rate=t.suggest_float("learning_rate", 0.005, 0.3, log=True),
            num_leaves=t.suggest_int("num_leaves", 4, 128, log=True),
            max_depth=t.suggest_int("max_depth", 3, 12),
            min_child_samples=t.suggest_int("min_child_samples", 5, 200, log=True),
            subsample=t.suggest_float("subsample", 0.4, 1.0),
            colsample_bytree=t.suggest_float("colsample_bytree", 0.2, 1.0),
            reg_lambda=t.suggest_float("reg_lambda", 0.001, 100, log=True),
            reg_alpha=t.suggest_float("reg_alpha", 0.001, 10, log=True),
            monotone_constraints_method=t.suggest_categorical(
                "monotone_constraints_method", ["basic", "intermediate", "advanced"]
            ),
        )
    if name == "CatBoost":
        return dict(
            iterations=t.suggest_int("iterations", 100, 1000, step=50),
            learning_rate=t.suggest_float("learning_rate", 0.005, 0.3, log=True),
            depth=t.suggest_int("depth", 2, 8),
            l2_leaf_reg=t.suggest_float("l2_leaf_reg", 0.1, 50, log=True),
            random_strength=t.suggest_float("random_strength", 0.01, 10, log=True),
            bootstrap_type="Bernoulli",
            subsample=t.suggest_float("subsample", 0.4, 1.0),
            rsm=t.suggest_float("rsm", 0.2, 1.0),
        )
    if name == "HistGradientBoosting":
        return dict(
            max_iter=t.suggest_int("max_iter", 100, 1200, step=50),
            learning_rate=t.suggest_float("learning_rate", 0.005, 0.3, log=True),
            max_depth=t.suggest_int("max_depth", 2, 10),
            max_leaf_nodes=t.suggest_int("max_leaf_nodes", 8, 128, log=True),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 5, 200, log=True),
            l2_regularization=t.suggest_float(
                "l2_regularization", 0.001, 100, log=True
            ),
            max_features=t.suggest_float("max_features", 0.2, 1.0),
        )
    if name in ("RandomForest", "ExtraTrees"):
        params = dict(
            n_estimators=t.suggest_int("n_estimators", 200, 500, step=100),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 80, log=True),
            min_samples_split=t.suggest_int("min_samples_split", 2, 60, log=True),
            max_features=t.suggest_float("max_features", 0.1, 1.0),
            max_depth=t.suggest_int("max_depth", 3, 30),
        )
        if name == "RandomForest":
            params["max_samples"] = t.suggest_float("max_samples", 0.3, 1.0)
        return params
    return dict(alpha=t.suggest_float("alpha", 0.01, 10000, log=True))  # Ridge

## Step 2: The search
`search(name)` runs the study of one algorithm and saves its trials; when the file is already there it only reads it. The objective is the **joint score**: the mean of the two tests (`new cities` is the mean of two different 5-fold splits, `next summers` is one run). Optuna cannot be made exactly repeatable when many trials run in parallel, so a new run gives slightly different trials with the same picture.

In [ ]:
def search(name):
    path = SEARCH / f"{name}_{N_TRIALS}trials.csv"
    if path.exists():
        return pd.read_csv(path)

    def objective(trial):
        params = space(name, trial)
        make = lambda seed: CANDIDATES[name](params, seed)
        new_cities = np.mean(
            [score_new_cities(*predict_new_cities(make, seed)) for seed in (100, 101)]
        )
        next_summers = score_next_summers(predict_next_summers(make, 100))
        trial.set_user_attr("new_cities", float(new_cities))
        trial.set_user_attr("next_summers", float(next_summers))
        return (new_cities + next_summers) / 2

    sampler = optuna.samplers.TPESampler(
        seed=1, constant_liar=True, multivariate=True, n_startup_trials=32
    )
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(
        objective, n_trials=N_TRIALS, n_jobs=-1, catch=(Exception,)
    )  # n_jobs=-1: one trial per CPU core
    trials = study.trials_dataframe(
        attrs=("number", "value", "params", "user_attrs", "duration", "state")
    )
    SEARCH.mkdir(parents=True, exist_ok=True)
    trials.to_csv(path, index=False)
    return trials


searches = {name: search(name) for name in CANDIDATES}
summary = pd.DataFrame(
    {
        name: {
            "trials": int((t["state"] == "COMPLETE").sum()),
            "best joint": t["value"].max(),
            "new cities": t.loc[t["value"].idxmax(), "user_attrs_new_cities"],
            "next summers": t.loc[t["value"].idxmax(), "user_attrs_next_summers"],
            "hours of CPU": t["duration"]
            .apply(lambda d: pd.Timedelta(d).total_seconds())
            .sum()
            / 3600,
        }
        for name, t in searches.items()
    }
).T.sort_values("best joint", ascending=False)
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, t in searches.items():
    done = t[t["state"] == "COMPLETE"].sort_values("number")
    ax.plot(
        done["value"].cummax().to_numpy(),
        label=name,
        lw=2 if name != "Ridge" else 1,
        ls="-" if name != "Ridge" else "--",
    )
ax.set_xlabel("trial")
ax.set_ylabel("best joint score so far")
ax.set_ylim(1.1, summary["best joint"].max() + 0.03)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**What we learned (Part 3)**
- **Every search has levelled off well before the end of the run** (figure): more trials would not change the picture. With the default `N_TRIALS = 30` the best joint scores are XGBoost 1.334, HistGradientBoosting 1.307, CatBoost 1.307, LightGBM 1.307, ExtraTrees 1.305 and RandomForest 1.283; the linear baseline reaches 1.138. The first full run (150 trials each) gave the same picture with slightly different values (XGBoost 1.340, LightGBM 1.334, HistGradientBoosting 1.324, ExtraTrees 1.312, CatBoost 1.306, RandomForest 1.281).
- **The picture of the first version of this notebook has changed.** There, ExtraTrees won: the boosting algorithms had only 40 trials each and the choice used one test (new cities). With the same search for everybody and the second test, the boosting algorithms are ahead. On the **next-summers** test they score about 1.51 to 1.57 against 1.46 and 1.47 for the two forests; on the **new-cities** test all algorithms are close (1.06 to 1.14). The algorithms differ most in how well they predict a district they have seen before.
- **The searches did not cost the same:** at 30 trials HistGradientBoosting used 1.4 CPU hours, CatBoost 1.3, RandomForest 0.9, ExtraTrees 0.4 and XGBoost and LightGBM about 0.2, and LightGBM reaches almost the same score. (The first full run, 150 trials each, cost HistGradientBoosting 14.5 CPU hours, RandomForest 7.7, XGBoost 5.4, ExtraTrees 4.6, CatBoost 4.6 and LightGBM 1.2.) Because many trials run at the same time, a new run gives somewhat different trials with the same picture.

# <span style="color:#F57C00">Part 4: Choosing the Model</span>

## The rule is written down before the result
Optuna picks the best of `N_TRIALS` trials on a *noisy* score, so the best trial is a little too lucky (the "winner's curse"). To be fair we do this:
1. **Fresh seeds:** the 5 best trials of every algorithm are tested again on 4 new seeds (new splits, new random numbers); the best of the five is the configuration of that algorithm, and its 4 scores give a standard error.
2. **Rule:** the model with the best joint score wins; every model **within one standard error** of it is a tie. Among the tied models the one whose predicted *change* matches the real changes best is preferred (Step 2: correlation, and a difference smaller than its own standard error, 0.03, counts as a tie too); among those, the **smallest model** wins. A model that is smaller and faster is better to use (the app runs it for every click).
3. A **blend** (the plain average of the best three algorithms) is a candidate under the same rule.
4. **The exam 2023-2025 does not take part in the choice.** It is opened once in Part 6.

## Step 1: The best trials on fresh seeds

In [ ]:
FRESH_SEEDS = (200, 201, 202, 203)
TOP = 5  # trials per algorithm that are tested again


def params_of(row):
    # the settings of a row of the trials table, with the types the models expect
    params = {
        c.replace("params_", ""): row[c]
        for c in row.index
        if c.startswith("params_") and pd.notna(row[c])
    }
    whole = {
        "n_estimators",
        "max_depth",
        "num_leaves",
        "min_child_samples",
        "iterations",
        "depth",
        "max_iter",
        "max_leaf_nodes",
        "min_samples_leaf",
        "min_samples_split",
    }
    params = {
        k: (
            int(v)
            if k in whole
            else float(v) if isinstance(v, (int, float, np.number)) else v
        )
        for k, v in params.items()
    }
    return {**params, "bootstrap_type": "Bernoulli"} if "rsm" in params else params


def fresh_evaluation(name, params, seed):
    make = lambda s: CANDIDATES[name](params, s)
    pred_new, fold = predict_new_cities(make, seed)
    pred_next = predict_next_summers(make, seed)
    return dict(
        new=score_new_cities(pred_new, fold),
        next=score_next_summers(pred_next),
        pred_new=pred_new,
        fold=fold,
        pred_next=pred_next,
    )


tasks = []
for name, t in searches.items():
    best_trials = (
        t[t["state"] == "COMPLETE"].sort_values("value", ascending=False).head(TOP)
    )
    tasks += [
        (name, rank, params_of(row), seed)
        for rank, (_, row) in enumerate(best_trials.iterrows())
        for seed in FRESH_SEEDS
    ]
results = Parallel(n_jobs=-1, backend="threading")(
    delayed(fresh_evaluation)(name, params, seed) for name, rank, params, seed in tasks
)

fresh = pd.DataFrame(
    [
        {"algorithm": n, "rank": r, "seed": s, "joint": (res["new"] + res["next"]) / 2}
        for (n, r, p, s), res in zip(tasks, results)
    ]
)
best_rank = (
    fresh.groupby(["algorithm", "rank"])["joint"]
    .mean()
    .groupby("algorithm")
    .idxmax()
    .map(lambda index: index[1])
)  # the best of the 5
best_params = {
    n: params_of(
        searches[n][searches[n]["state"] == "COMPLETE"]
        .sort_values("value", ascending=False)
        .iloc[best_rank[n]]
    )
    for n in CANDIDATES
}
predictions = (
    {  # the predictions of the best configuration of every algorithm, for the blends
        n: {
            s: res
            for (name, r, p, s), res in zip(tasks, results)
            if name == n and r == best_rank[n]
        }
        for n in CANDIDATES
    }
)
print("settings of the best configuration of every algorithm:")
for name in CANDIDATES:
    print(f"  {name}: {best_params[name]}")
print(
    "joint score on fresh seeds of the 5 tested trials (row = place in the search, column = algorithm):"
)
fresh.groupby(["rank", "algorithm"])["joint"].mean().unstack().round(3)

In [ ]:
def joint_scores(members):
    # the joint score of the average of the members, for every fresh seed (a single algorithm is a blend of one)
    out = []
    for seed in FRESH_SEEDS:
        pred_new = np.mean([predictions[m][seed]["pred_new"] for m in members], axis=0)
        pred_next = np.mean(
            [predictions[m][seed]["pred_next"] for m in members], axis=0
        )
        out.append(
            (
                score_new_cities(pred_new, predictions[members[0]][seed]["fold"]),
                score_next_summers(pred_next),
            )
        )
    return np.array(out)


forest_and_boosting = [n for n in CANDIDATES if n != "Ridge"]
singles = {name: joint_scores([name]) for name in CANDIDATES}
top3 = sorted(forest_and_boosting, key=lambda n: -singles[n].mean())[:3]
candidates = {**singles, "Blend of " + " + ".join(top3): joint_scores(top3)}
comparison = pd.DataFrame(
    {
        name: {
            "new cities": s[:, 0].mean(),
            "next summers": s[:, 1].mean(),
            "joint": s.mean(axis=1).mean(),
            "sem": s.mean(axis=1).std(ddof=1)
            / np.sqrt(len(s)),  # standard error over the 4 fresh seeds
        }
        for name, s in candidates.items()
    }
).T.sort_values("joint", ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.errorbar(
    comparison["joint"],
    comparison.index,
    xerr=comparison["sem"],
    fmt="o",
    color="#F57C00",
    capsize=4,
)
ax.invert_yaxis()
ax.set_xlabel(
    "joint score on fresh seeds (higher is better; the line is one standard error)"
)
plt.tight_layout()
plt.show()
comparison.round(3)

## Step 2: Does the predicted *change* match real changes?
The what-if is about changes, so the second criterion is the check of the what-if notebook (Part 4 there), here **inside the training years**: districts whose green share changed between 2013-2015 and 2020-2022 against how much their surface temperature changed, both minus the city mean. Each model was trained on the years up to 2019 only, so it has not seen the late period. `corr` is the correlation between the change that the model attributes to the change of the green share and the observed change; `attributed` is the cooling per +10 points that it attributes to these districts (the observed value is the reference). The size is the model file on disk.

In [ ]:
periods = {"early": (2013, 2015), "late": (2020, 2022)}
means = pd.concat(
    {
        name: table[table["year"].between(*years)]
        .groupby("district_code")
        .agg(
            n_years=("year", "size"),
            green=("landsat_green_share", "mean"),
            lst=(TARGET, "mean"),
        )
        for name, years in periods.items()
    },
    axis=1,
).dropna()
means.columns = [f"{name}_{period}" for period, name in means.columns]
means = means[(means["n_years_early"] >= 2) & (means["n_years_late"] >= 2)]
means["city_code"] = table.groupby("district_code")["city_code"].first()
changes = means.assign(
    d_green=means["green_late"] - means["green_early"],
    d_lst=means["lst_late"] - means["lst_early"],
)
changes = changes[changes.groupby("city_code")["d_green"].transform("size") >= 3].copy()
for name in ("d_green", "d_lst"):
    changes[f"{name}_c"] = changes[name] - changes.groupby("city_code")[name].transform(
        "mean"
    )  # minus the city mean
observed = stats.linregress(changes["d_green_c"], changes["d_lst_c"]).slope * 0.10

late_rows = table[table["year"].between(*periods["late"])].copy()
swapped = late_rows.copy()  # the late rows with the early green share
swapped["landsat_green_share"] = (
    swapped["district_code"]
    .map(changes["green_early"])
    .fillna(swapped["landsat_green_share"])
)
swapped = add_within_city(
    swapped, WITHIN_CITY
)  # whole city-years are in the table, so the within-city columns are right

before_late = table[table["year"] < periods["late"][0]]
model_changes = (
    {}
)  # the change per district that each model attributes to the change of the green share
for name in forest_and_boosting:
    models = [
        CANDIDATES[name](best_params[name], seed).fit(
            before_late[FEATURES], before_late[TARGET]
        )
        for seed in (0, 1, 2)
    ]  # 3 seeds, averaged
    change = np.mean(
        [m.predict(late_rows[FEATURES]) - m.predict(swapped[FEATURES]) for m in models],
        axis=0,
    )
    model_changes[name] = (
        pd.Series(change, index=late_rows.index)
        .groupby(late_rows["district_code"])
        .mean()
    )
blend = "Blend of " + " + ".join(top3)
model_changes[blend] = pd.concat([model_changes[n] for n in top3], axis=1).mean(
    axis=1
)  # a blend attributes the average of its members


def real_change_stats(by_district):
    c = changes.assign(model=by_district)
    c["model_c"] = c["model"] - c.groupby("city_code")["model"].transform("mean")
    return dict(
        attributed=np.polyfit(c["d_green_c"], c["model_c"], 1)[0] * 0.10,
        corr=stats.pearsonr(c["model_c"], c["d_lst_c"])[0],
    )


def model_size_mb(name, params):
    # the size of the model file in the native format of its library
    model = CANDIDATES[name](params, 0).fit(X_train, y_train)
    MODELS.mkdir(exist_ok=True)
    probe = MODELS / "_size_probe"
    if name == "XGBoost":
        model.save_model(f"{probe}.ubj")
    elif name == "LightGBM":
        model.booster_.save_model(f"{probe}.txt")
    elif name == "CatBoost":
        model.save_model(f"{probe}.cbm")
    else:
        joblib.dump(model, f"{probe}.joblib")
    file = next(MODELS.glob("_size_probe.*"))
    size = file.stat().st_size / 1e6
    file.unlink()
    return size


sizes = dict(
    zip(
        forest_and_boosting,
        [model_size_mb(n, best_params[n]) for n in forest_and_boosting],
    )
)
sizes[blend] = sum(sizes[n] for n in top3)  # a blend needs all its members
profile = pd.DataFrame(
    {
        name: {**real_change_stats(by_district), "size_mb": sizes[name]}
        for name, by_district in model_changes.items()
    }
).T
print(
    f"{len(changes)} districts changed their green share between the two periods; observed: {observed:+.2f} °C per +10 points"
)
profile.round(3)

## Step 3: The rule and the winner

In [ ]:
table_c = comparison.join(profile).drop(
    index="Ridge"
)  # the baseline is not a candidate
best = table_c["joint"].idxmax()
tie = table_c[
    table_c["joint"] >= table_c.loc[best, "joint"] - table_c.loc[best, "sem"]
]  # within one standard error of the best
tie = tie[
    tie["corr"] >= tie["corr"].max() - 0.03
]  # the correlation of ~800 districts has a standard error of about 0.03
winner = tie.sort_values("size_mb").index[0]
assert (
    winner in CANDIDATES
), "a blend won: this notebook then needs the code to train every member; see the comparison table"

print(
    f"best joint score: {best} ({table_c.loc[best, 'joint']:.3f} +- {table_c.loc[best, 'sem']:.3f})"
)
print("tied with it:", list(tie.index))
print(
    f"WINNER: {winner}, the smallest of the tied models ({table_c.loc[winner, 'size_mb']:.1f} MB)"
)
print("settings:", best_params[winner])
table_c.round(3)

**What we learned (Part 4)**
- **On fresh seeds the order holds:** XGBoost 1.351 ± 0.004, HistGradientBoosting 1.327, LightGBM 1.326, CatBoost 1.322, ExtraTrees 1.316, RandomForest 1.291, Ridge 1.148. XGBoost is ahead of ExtraTrees by about eight standard errors, so this is no noise. The five tested trials of every algorithm score within a narrow band (table), so the result does not depend on one lucky trial.
- **New cities against next summers:** for new cities the tree algorithms are close (1.09 to 1.16); for the next summers of known districts the boosting algorithms are clearly better (1.54 to 1.57 against 1.46 and 1.48 for the forests). The app serves known districts, so the second test is the one that matters most for it.
- **A blend of the best three algorithms** scores 1.347, a little *below* XGBoost alone: not worth carrying three models.
- **The real-change check inside the training years:** the 824 districts that changed their green share between 2013-2015 and 2020-2022 cooled by **-0.60 °C per +10 points** relative to their city. Every model attributes a *steeper* cooling to these changes: the two forests the least (-0.65 and -0.64), then CatBoost -0.73, XGBoost -0.75, LightGBM -0.76 and HistGradientBoosting -0.82. The correlations with the observed change (0.26 to 0.31) differ by about their standard error (0.03) and do not separate the models. **XGBoost's what-if is about 1.3 times the observed change**, so its numbers should be read together with the range of the what-if notebook.
- **The rule gives XGBoost:** within one standard error of the best score are XGBoost and the blend, their correlations are tied, and XGBoost is the smaller of the two (5.4 MB against 13.4 MB). LightGBM is only 3.0 MB but about seven standard errors behind; ExtraTrees is by far the largest model (169 MB).
- **The settings of the production model are pinned:** 650 trees of depth 10 with a learning rate of 0.009, each tree seeing about half of the rows and half of the features — the configuration that the *original* full search chose, so that the saved file is exactly the one the application serves. Parts 5 and 6 use it. This lighter search picked a shallower, faster-learning XGBoost (900 trees of depth 7, learning rate 0.016) with the same fresh-seed picture — the choice of XGBoost is not one lucky corner.

# <span style="color:#F57C00">Part 5: Improving the Model: Two More Feature Groups and the Air Temperature</span>

Part 4 chose **XGBoost** and its settings. The deep search tuned the settings, not the *information* the model gets or the *way* it uses it. This part asks two more questions with the same two tests (and the same rule: a change must be **better with a margin that is not noise**, otherwise it is not made):
1. Which **new information** helps? 2. Does it help to give the model the **air temperature of the summer** directly?

## Ideas that were tested and did not help
These were tried in a separate probe (outside this notebook) with the same two tests and a bootstrap over cities. The numbers are the change of the joint score against the model of Part 4 (a difference smaller than about 0.01 is noise):

| Idea | Result |
| :--- | :--- |
| The green share of the earlier summers (mean of the 3 summers before) | -0.007: nothing; the model already has the green share of the summer itself |
| A 3-year smoothed green share (less measurement noise), also instead of the raw one | -0.005, and -0.025 when it replaces the raw share |
| The weather of the summer before, extra weather (maximum temperature, day-night range, anomalies, rain of the winter and of the whole water year) | helps new cities, **hurts known districts** (within-city R² of the next summers -0.007 to -0.031) |
| The district's own surface temperature of the earlier summers (lag) | the balance does not change and the RMSE gets worse |
| Weights by pixel count; every month as its own row; NDVI as a feature; permits and population growth | no gain; NDVI also breaks the what-if (it changes the green share but cannot change the NDVI together with it). *Tested in the independent review of this project and not repeated here.* |

**What stays:** the surface temperature of the **earlier summers is not needed**: after the model has the right climate of the district, the remaining error is a persistent offset of each district (53% of it) and a shock of a whole city in one summer (41%) that no feature of ours explains. This is a limit of the data, not of the algorithm.

## Step 1: Two feature groups that help
- **Clear sky** (`landsat_cover`, and its within-city version): the share of clear pixels of the summer. Rows with a cloud remnant read about 0.3 to 0.5 °C too cool (EDA, part C5); the model can correct this with the cover. It is a *measurement condition*, not a cause: in the what-if it stays as it is.
- **Climate normals** (`precip_clim`, `temp_clim`): mean annual rain and mean temperature of 2000-2012, the years **before** the first Landsat summer. They describe the climate of the district without using any year of the model table, so there is no leakage.

In [ ]:
WITHIN_CITY_NEW = WITHIN_CITY + [
    "landsat_cover"
]  # the within-city columns of the final model also have the clear sky

early = climate[climate["year"] <= 2012]  # before the first summer of the model table
normals = pd.DataFrame(
    {
        "precip_clim": early.groupby(["district_code", "year"])["precip_sum"]
        .sum()
        .groupby("district_code")
        .mean(),  # mean annual rain
        "temp_clim": early.groupby("district_code")["temp_mean"].mean(),
    }
).reset_index()
table = add_within_city(
    table.merge(normals, on="district_code", how="left", validate="many_to_one"),
    WITHIN_CITY_NEW,
)  # the other within-city columns are calculated again, with the same values

ADDED_FEATURES = ["landsat_cover", "landsat_cover_vs_city", "precip_clim", "temp_clim"]
FEATURES_NEW = FEATURES + ADDED_FEATURES
assert table[FEATURES_NEW].notna().all().all() and len(table) == 10622
train = table[table["year"] <= TRAIN_END].reset_index(
    drop=True
)  # the same rows as before, with the new columns
test = table[table["year"] > TRAIN_END].reset_index(drop=True)
print(
    f"{len(FEATURES)} features of Part 4 + {len(ADDED_FEATURES)} new = {len(FEATURES_NEW)}"
)
table[ADDED_FEATURES].describe().round(2)

## Step 2: Giving the model the air temperature
The surface temperature of a district follows the air temperature of the summer. A tree model predicts values that it has seen in the training data, so for a summer that is hotter than the earlier ones it predicts too cool (the bias of the exam in Part 6). The remedy is to let the trees learn the **surface temperature minus half of the air temperature** (`temp_mean_ja`, the mean air temperature of July and August) and to add that half back when predicting. A hotter summer then moves the prediction directly, and the trees only have to learn the rest.

`AirTemperatureOffset` does this around any scikit-learn model (it is a regular scikit-learn estimator: `fit`, `predict`, `clone`, and tools like `permutation_importance` work with it). The green share stays monotone, and in the what-if the air temperature does not change when the green share changes.

**Why half?** In the probe the share was tried with 0.5, 0.75, 1 and 1.25 of the air temperature (a faster model, the same tests): the joint scores were 1.371, 1.366, 1.367 and 1.348, against 1.363 without it. So some share helps and 0.5 is the best of the four, but the scores are close; the share is the one choice that was made with a few values on the training years, and the fresh seeds below confirm it.

In [ ]:
GREEN = (
    "landsat_green_share",
    "landsat_green_share_vs_city",
)  # more green must never mean a hotter surface


def make_xgb(features, params, seed=0, n_jobs=1):
    # XGBoost with the settings of Part 4; the green share may only lower the prediction
    monotone = tuple(-1 if name in GREEN else 0 for name in features)
    return XGBRegressor(
        **params,
        monotone_constraints=monotone,
        tree_method="hist",
        n_jobs=n_jobs,
        random_state=seed
    )


class AirTemperatureOffset(RegressorMixin, BaseEstimator):
    """The final model. `estimator` learns the surface temperature minus a share of the air temperature of the summer;
    the share is added back when predicting. A hotter summer then moves the prediction directly, which trees cannot do
    on their own beyond the summers they have seen. `RegressorMixin` must come first: scikit-learn only recognizes a
    class as a regressor in that order, and cross-validation helpers check it. The state is explicit both ways:
    `fit` sets `estimator_`, and `from_loaded` sets it for a model that was read from a file."""

    def __init__(self, estimator, share=0.5, column="temp_mean_ja"):
        self.estimator, self.share, self.column = estimator, share, column

    def fit(self, X, y):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        self.estimator_ = clone(self.estimator).fit(X, y - self.share * X[self.column])
        return self

    def predict(self, X):
        estimator = getattr(self, "estimator_", None)  # a model loaded from a file has no fitted state yet
        if estimator is None:
            raise ValueError("the model is neither fitted nor loaded")
        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError("the prediction columns differ from the fitted model")
        return estimator.predict(X) + self.share * X[self.column].to_numpy()

    @classmethod
    def from_loaded(cls, estimator, features, share=0.5, column="temp_mean_ja"):
        """A native XGBoost file read from disk, with its fitted state made explicit."""
        booster_names = estimator.get_booster().feature_names
        if booster_names is not None and list(booster_names) != list(features):
            raise ValueError("the model file uses other feature names than the model card")
        model = cls(estimator, share, column)
        model.estimator_ = estimator
        model.feature_names_in_ = np.asarray(features, dtype=object)
        model.n_features_in_ = len(features)
        return model


def new_final(features, params, seed=0, n_jobs=1):
    return AirTemperatureOffset(make_xgb(features, params, seed, n_jobs))


def load_final(card, folder):
    # the final model from the files of the model card, without training
    trees = XGBRegressor()
    trees.load_model(folder / card["model_file"])  # the native format of XGBoost
    return AirTemperatureOffset.from_loaded(
        trees, card["features"], card["air_temperature_share"],
        card["air_temperature_column"],
    )

## Step 3: The tests on fresh seeds
Two models are evaluated with the two tests of Part 2 on the same 4 fresh seeds as in Part 4 (`predict_new_cities` and `predict_next_summers` take the list of columns): the **direct** model with the 30 features, and the **final** model (the same trees around the air temperature offset). The stored predictions of Part 4 are reused for the model **before** (XGBoost, 26 features). **The settings are pinned** to the configuration that the original full search selected, so the saved model does not change when the lighter search picks slightly different trials: the file this notebook writes is exactly the file the application serves.

In [ ]:
# The settings of the production model: chosen by the original deep search (Part 4 of the first
# full run, 150 trials per algorithm) and pinned here, so the saved model stays exactly the
# file the application serves. The lighter search above is a demonstration; a re-run may pick
# slightly different trials but never changes this model.
PARAMS = {
    "colsample_bytree": 0.4782728394917927,
    "gamma": 0.0010460965175023,
    "learning_rate": 0.0093206033695145,
    "max_depth": 10,
    "min_child_weight": 1.5528224543048317,
    "n_estimators": 650,
    "reg_alpha": 0.0293907050647303,
    "reg_lambda": 0.0269922583455769,
    "subsample": 0.4838401077550928,
}
new_models = {
    "direct": lambda seed: make_xgb(FEATURES_NEW, PARAMS, seed),
    "final": lambda seed: new_final(FEATURES_NEW, PARAMS, seed),
}


def new_model_evaluation(name, seed):
    pred_new, fold = predict_new_cities(new_models[name], seed, FEATURES_NEW)
    return dict(
        pred_new=pred_new,
        fold=fold,
        pred_next=predict_next_summers(new_models[name], seed, FEATURES_NEW),
    )


tasks5 = [(name, seed) for name in new_models for seed in FRESH_SEEDS]
results5 = Parallel(n_jobs=-1, backend="threading")(
    delayed(new_model_evaluation)(name, seed) for name, seed in tasks5
)
for name in new_models:
    predictions[name] = {
        seed: res for (n, seed), res in zip(tasks5, results5) if n == name
    }  # joint_scores() of Part 4 works with these too

steps = {
    "before: XGBoost, 26 features (Part 4)": "XGBoost",
    "+ clear sky and climate normals (direct)": "direct",
    "+ half of the air temperature (the final model)": "final",
}
improvement = {}
for step, name in steps.items():
    s = joint_scores([name])  # one row per fresh seed: new cities, next summers
    improvement[step] = {
        "new cities": s[:, 0].mean(),
        "next summers": s[:, 1].mean(),
        "joint": s.mean(axis=1).mean(),
        "sem": s.mean(axis=1).std(ddof=1) / np.sqrt(len(s)),
        "RMSE new cities": np.mean(
            [
                scores(train, predictions[name][seed]["pred_new"])["rmse"]
                for seed in FRESH_SEEDS
            ]
        ),
        "RMSE next summers": np.mean(
            [
                scores(next_rows, predictions[name][seed]["pred_next"])["rmse"]
                for seed in FRESH_SEEDS
            ]
        ),
    }
improvement = pd.DataFrame(improvement).T
improvement.round(3)

## Step 4: Is the gain real? A bootstrap over cities
The seeds change the splits, but they do not change *which cities exist*. To see whether a gain could be luck of the 80 cities, the cities are **drawn again and again with replacement** (1,000 draws, like in the what-if notebook): every draw scores the same predictions again, with a city that is drawn twice counting twice (the `sample_weight` of the scikit-learn metrics). A gain counts when its **95% interval stays above 0**. The scores of the two tests are pooled over the folds and the years, so they differ a little from the means of Step 3; the differences are what matters. The last two columns are the within-city R² (which the what-if depends on) and the RMSE (positive = the new model is better).

In [ ]:
cities = np.sort(train["city_code"].unique())
counts = np.random.default_rng(0).multinomial(
    len(cities), np.ones(len(cities)) / len(cities), size=1000
)
counts = np.vstack(
    [np.ones(len(cities)), counts]
)  # row 0: every city once = the plain score


def sample_scores(rows, predicted):
    # balance, within-city R2 and RMSE of the predictions for every sample of cities (rows of `counts`)
    r = rows[["city_code", "year", TARGET]].assign(predicted=np.asarray(predicted))
    grouped = r.groupby(["city_code", "year"])[[TARGET, "predicted"]]
    level = grouped.mean().reset_index()  # one row per city and year
    inside = r[[TARGET, "predicted"]] - grouped.transform(
        "mean"
    )  # the part inside the cities
    city_of_level, city_of_row = np.searchsorted(
        cities, level["city_code"]
    ), np.searchsorted(cities, r["city_code"])
    out = []
    for weights in counts:  # how often every city is in this sample
        w_level, w_row = weights[city_of_level], weights[city_of_row]
        r2_within = r2_score(inside[TARGET], inside["predicted"], sample_weight=w_row)
        r2_level = r2_score(level[TARGET], level["predicted"], sample_weight=w_level)
        out.append(
            [
                r2_level + r2_within,
                r2_within,
                root_mean_squared_error(r[TARGET], r["predicted"], sample_weight=w_row),
            ]
        )
    return np.array(out)


tests5 = (("new cities", train, "pred_new"), ("next summers", next_rows, "pred_next"))
sampled = {  # model -> test -> scores of every sample, the mean over the fresh seeds
    name: {
        test_name: np.mean(
            [sample_scores(rows, predictions[name][seed][key]) for seed in FRESH_SEEDS],
            axis=0,
        )
        for test_name, rows, key in tests5
    }
    for name in ("XGBoost", "direct", "final")
}


def gain(better, baseline, test_name):
    # (draws x [balance, within-city R2, RMSE]); the RMSE is lower when it is better, so its gain is baseline minus better
    a, b = sampled[baseline][test_name], sampled[better][test_name]
    return np.column_stack([b[:, 0] - a[:, 0], b[:, 1] - a[:, 1], a[:, 2] - b[:, 2]])


rows_out = []
for better, baseline in (
    ("direct", "XGBoost"),
    ("final", "XGBoost"),
    ("final", "direct"),
):
    g = {t: gain(better, baseline, t) for t in ("new cities", "next summers")}
    g["joint"] = (g["new cities"] + g["next summers"]) / 2
    for test_name in ("joint", "new cities", "next summers"):
        low, high = np.percentile(g[test_name][1:, 0], [2.5, 97.5])
        rows_out.append(
            {
                "comparison": f"{better} against {baseline}",
                "test": test_name,
                "gain of the balance": g[test_name][0, 0],
                "95% from": low,
                "95% to": high,
                "within-city R2": g[test_name][0, 1],
                "RMSE": g[test_name][0, 2],
                "interval above 0": low > 0,
            }
        )
gain_table = pd.DataFrame(rows_out).set_index(["comparison", "test"])
gain_table.round(3)

In [ ]:
# the rule, written before the result: the final model must beat the model of Part 4 with an interval above 0 and must not lose within-city R2
final_against_before = gain_table.loc["final against XGBoost"]
assert final_against_before[
    "interval above 0"
].all(), "the gain is not clearly above 0 in every test"
assert (
    final_against_before["within-city R2"] > -0.02
).all(), "the within-city R2 loses more than 0.02"
joint = final_against_before.loc["joint"]
print(
    f"the final model beats the model of Part 4: joint gain {joint['gain of the balance']:+.3f} (95% {joint['95% from']:+.3f} to {joint['95% to']:+.3f})"
)

## A simpler model on purpose
The average of **three** views was tested too, in the same probe (the trees on the surface temperature, the trees on the surface temperature minus half of the air temperature, and a straight line plus trees). It is a little better than the single final model, but it needs three models, so the single model was chosen for **simplicity**. The numbers (fresh seeds, walk-forward exam, bootstrap of the what-if), for the record:

| | Before (Part 4) | Direct, 30 features | **Final: half of the air temperature** | Average of three views |
| :--- | :---: | :---: | :---: | :---: |
| Joint score, fresh seeds | 1.357 | 1.391 | **1.396** | 1.410 |
| RMSE of the next summers | 1.451 | 1.410 | 1.365 | 1.356 |
| Exam RMSE 2023-2025 | 1.454 | 1.334 | **1.274** | 1.252 |
| Exam RMSE, weather scenario | 1.740 | 1.580 | **1.558** | 1.584 |
| What-if, +10 points | -0.72 | -0.65 | -0.62 | -0.60 |
| Width of the bootstrap range at +10 points | 0.48 | 0.43 | 0.43 | **0.36** |
| Models, file size | 1, 16.6 MB | 1, 16.5 MB | 1, 16.5 MB | 3, 46 MB |

The three views are better by +0.013 (95% +0.006 to +0.021) on the joint score and give a narrower what-if range; the single model has about 75% of their gain in the joint score and about 90% of it in the exam, is as good under the weather scenario, and has one model, one file and one small class. (Probe numbers with the average of 3 seeds; the final model of this notebook is trained with one seed, so the last digit can differ.)

**What we learned (Part 5)**
- **The new information helps new cities most.** Clear sky and the climate normals raise the joint score from 1.351 to 1.390: **+0.039** (95% interval +0.026 to +0.054); +0.066 (+0.046 to +0.093) in the new-cities test, but only +0.011 (-0.002 to +0.025) for the next summers of known districts, which already "know" their own climate. The RMSE falls from 1.79 to 1.71 °C for new cities and from 1.45 to 1.41 °C for next summers.
- **The air temperature offset helps known districts.** On top of the direct model it adds +0.018 (+0.013 to +0.024) in the next-summers test and nothing for new cities (-0.004, interval -0.018 to +0.010); the joint gain is small, **+0.007** (0.000 to +0.014). The RMSE of the next summers falls from 1.41 to 1.36 °C, and Part 6 shows that the model predicts less too cool in the hot summers of the exam. The step is modest in the tests of the training years and larger in the exam, where the summers are hotter than the training years.
- **Together:** the joint score goes from 1.351 ± 0.004 to **1.397 ± 0.003** (+0.046, 95% interval +0.032 to +0.062, so the gain is no luck of the 80 cities): +0.063 (+0.040 to +0.089) for new cities and +0.029 (+0.016 to +0.043) for the next summers. The RMSE is 0.08 to 0.09 °C lower in both tests, and the within-city R² (the part the what-if is about) is **higher** (+0.030: +0.047 for new cities, +0.013 for the next summers), not lower. The rule above passed.
- **The price is nothing:** one model, one file of 16 MB and one small class; the final model is as simple to use as the model of Part 4.

# <span style="color:#F57C00">Part 6: The Exam on the Hidden Years and the Final Model</span>

Now the test years 2023, 2024 and 2025 are opened, and the choice of Part 5 does not change any more.

**A note on honesty.** These years were opened before: for the first version of this notebook (ExtraTrees), in a quick look at an earlier XGBoost, and again for the probe of Part 5, in two forms (one model for all three years and the walk-forward below). All choices were made with the training years only and the finalists were written down before the exam was opened, but a reader should know that the exam is not untouched in the history of this project.

## Step 1: Walk-forward exam
For each test year the model is trained on **all earlier years** and predicts that year, so no year is predicted by a model that has seen it. Every year is predicted in two ways:
- with the **real weather** of that summer (the task of the model);
- with a **weather scenario**: the mean of the previous 5 summers of the district. For a future summer nobody knows the weather, so this is the simple scenario of the project; the difference between the two shows what the weather uncertainty costs.

The final model is compared with **the model before Part 5** (XGBoost, 26 features) and with two simple baselines: **Ridge** and **the mean of the district's own past summers** (a baseline that only exists for districts that were seen before).

In [ ]:
# Scenario weather: the mean of the 5 summers before, per district (shift(1) = without the year itself)
weather = weather.sort_values(["district_code", "year"])
scenario_weather = weather[["district_code", "year"]].copy()
for name in WEATHER:
    scenario_weather[name] = weather.groupby("district_code")[name].transform(
        lambda s: s.shift(1).rolling(5).mean()
    )

test_years = sorted(test["year"].unique())

predictions_exam, effects = [], []
for year in test_years:
    earlier, current = table[table["year"] < year], table[table["year"] == year]
    model = new_final(FEATURES_NEW, PARAMS, 0, n_jobs=-1).fit(
        earlier[FEATURES_NEW], earlier[TARGET]
    )
    before = CANDIDATES["XGBoost"](PARAMS, 0).fit(
        earlier[FEATURES], earlier[TARGET]
    )  # the model of Part 4
    ridge_model = CANDIDATES["Ridge"](best_params["Ridge"], 0).fit(
        earlier[FEATURES], earlier[TARGET]
    )
    past_mean = earlier.groupby("district_code")[TARGET].mean()

    scenario = current[["district_code", "year"]].merge(
        scenario_weather, on=["district_code", "year"], how="left"
    )
    scenario_rows = current.copy()
    scenario_rows[WEATHER] = scenario[WEATHER].to_numpy()
    scenario_rows = add_within_city(
        scenario_rows, WITHIN_CITY_NEW
    )  # the within-city weather columns change with the weather

    predictions_exam.append(
        current[["district_code", "city_code", "district_name", "year", TARGET]].assign(
            model_real_weather=model.predict(current[FEATURES_NEW]),
            model_weather_scenario=model.predict(scenario_rows[FEATURES_NEW]),
            before_real_weather=before.predict(current[FEATURES]),
            before_weather_scenario=before.predict(scenario_rows[FEATURES]),
            ridge=ridge_model.predict(current[FEATURES]),
            district_past_mean=current["district_code"].map(past_mean).to_numpy(),
            region=current["city_code"].map(region_of_city),
        )
    )

    # what-if: +10 points of green share in ONE district at a time (the city mean moves by 1/n of the change).
    # The same policy as the what-if notebook: the new share is clipped to the range the training rows of
    # this fold have seen, so the model is never asked about a share it has not seen, and a district already
    # above that maximum keeps its share (it only moves the city mean).
    changed = current.copy()
    n_districts = changed.groupby(["city_code", "year"])["district_code"].transform(
        "size"
    )
    upper = np.maximum(
        earlier["landsat_green_share"].max(), current["landsat_green_share"]
    )
    changed["landsat_green_share"] = np.clip(
        current["landsat_green_share"] + 0.10, 0, upper
    )
    applied = changed["landsat_green_share"] - current["landsat_green_share"]
    changed["landsat_green_share_vs_city"] += applied * (1 - 1 / n_districts)
    effects.append(
        pd.DataFrame(
            {
                "region": current["city_code"].map(region_of_city),
                "effect": model.predict(changed[FEATURES_NEW])
                - model.predict(current[FEATURES_NEW]),
            }
        )
    )

exam = pd.concat(predictions_exam, ignore_index=True)
names = {
    "model_real_weather": "Final model, real weather",
    "model_weather_scenario": "Final model, weather scenario",
    "before_real_weather": "Before Part 5, real weather",
    "before_weather_scenario": "Before Part 5, weather scenario",
    "ridge": "Ridge",
    "district_past_mean": "district's own past mean",
}
exam_scores = {}
for column, name in names.items():
    exam_scores[name] = {
        str(year): scores(
            exam[exam["year"] == year], exam.loc[exam["year"] == year, column]
        )
        for year in test_years
    }
    exam_scores[name]["2023-2025"] = scores(exam, exam[column])
exam_table = pd.concat(
    {name: pd.DataFrame(by_year).T for name, by_year in exam_scores.items()}
)
exam_table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
rmse = (
    exam_table["rmse"]
    .unstack(level=1)[["2023", "2024", "2025", "2023-2025"]]
    .loc[list(names.values())]
)
rmse.plot.bar(
    ax=axes[0], rot=20, colormap="Oranges_r", edgecolor="black", linewidth=0.5
)
axes[0].set_ylabel("RMSE (°C), lower is better")
axes[0].set_xlabel("")
axes[0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False)

latest = exam[exam["year"] == test_years[-1]]
axes[1].scatter(
    latest[TARGET], latest["model_real_weather"], s=6, alpha=0.5, color="#2c6e91"
)
limits = [latest[TARGET].min() - 1, latest[TARGET].max() + 1]
axes[1].plot(limits, limits, color="black", lw=0.8)
axes[1].set_xlabel("measured surface temperature (°C)")
axes[1].set_ylabel("predicted (°C)")
axes[1].set_title(f"{test_years[-1]}: every point is a district")
plt.tight_layout()
plt.show()
for column in ("before_real_weather", "model_real_weather"):
    print(
        f"mean error (measured minus predicted) by year, {names[column]}: {(exam[TARGET] - exam[column]).groupby(exam['year']).mean().round(2).to_dict()}"
    )

## Step 2: What-if check
What does the model say when the green share of a district rises by **10 points**, everything else unchanged? The measured relation inside the cities is about -0.6 °C (EDA, part E1). The check uses the model of each test year; `effect` is the predicted change in °C.

In [ ]:
effects = pd.concat(effects)
print(
    f"+10 points of green share: mean change {effects['effect'].mean():+.2f} °C | districts that get cooler: {(effects['effect'] < 0).mean():.1%}"
)
by_region = effects.groupby("region")["effect"].mean().sort_values()
print("by region:", by_region.round(2).to_dict())
ax = by_region.plot.barh(figsize=(7, 3.2), color="#F57C00")
ax.set_xlabel("mean change of the surface temperature for +10 points of green (°C)")
plt.tight_layout()
plt.show()

## Step 3: Which features matter?
Permutation importance (a scikit-learn tool): one feature at a time is shuffled (5 times) and we see how much the error of the 2025 exam grows. A big growth means that the model needs the feature. (`model` is the final model of the last test year, trained on 2013-2024.)

In [ ]:
latest_rows = table[table["year"] == test_years[-1]]
importance = permutation_importance(
    model,
    latest_rows[FEATURES_NEW],
    latest_rows[TARGET],
    n_repeats=5,
    random_state=0,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
top = (
    pd.Series(importance.importances_mean, index=FEATURES_NEW)
    .sort_values(ascending=False)
    .head(12)
)
ax = top.sort_values().plot.barh(figsize=(7, 4), color="#F57C00")
ax.set_xlabel("increase of the RMSE (°C) when the feature is shuffled")
plt.tight_layout()
plt.show()

## Step 4: The final model on all years
The exam is done, so the model is now trained on **all years 2013-2025** with the chosen settings and saved in the **native format of XGBoost** (one `.ubj` file: small, and it loads without scikit-learn). The air temperature share is in the model card, because it is not part of the file. We also save the exam predictions (for the error band of the what-if notebook). The final model is **not scored again** (all scores above come from models that did not see the year they were tested on). A last check loads the file again and compares the predictions.

In [ ]:
MODELS.mkdir(exist_ok=True)
final_model = new_final(FEATURES_NEW, PARAMS, 0, n_jobs=-1).fit(
    table[FEATURES_NEW], table[TARGET]
)
final_model.estimator_.save_model(
    MODELS / "final_model.ubj"
)  # the native format of XGBoost
exam.to_csv(MODELS / "exam_predictions.csv", index=False)

model_card = {
    "algorithm": "XGBoost on the surface temperature minus half of the air temperature",
    "model_file": "final_model.ubj",
    "air_temperature_share": final_model.share,
    "air_temperature_column": final_model.column,
    "settings": PARAMS,
    "features": FEATURES_NEW,
    "monotone_constraints": {name: -1 if name in GREEN else 0 for name in FEATURES_NEW},
    "trained_on_years": [int(table["year"].min()), int(table["year"].max())],
    "training_rows": len(table),
    "artifact_schema_version": 2,
    "model_sha256": hashlib.sha256((MODELS / "final_model.ubj").read_bytes()).hexdigest(),
    "evaluation_label": "Historical walk-forward; the 2023-2025 years were inspected in earlier development",
    "search": {
        "trials_per_algorithm": N_TRIALS,
        "joint_scores_on_fresh_seeds": comparison["joint"].round(4).to_dict(),
    },
    "joint_scores_of_the_steps_of_part_5": improvement["joint"].round(4).to_dict(),
    "exam_2023_2025_real_weather": exam_scores["Final model, real weather"]["2023-2025"]
    .round(4)
    .to_dict(),
    "exam_2023_2025_before_part_5": exam_scores["Before Part 5, real weather"][
        "2023-2025"
    ]
    .round(4)
    .to_dict(),
    "exam_2023_2025_weather_scenario": exam_scores["Final model, weather scenario"][
        "2023-2025"
    ]
    .round(4)
    .to_dict(),
    "what_if_plus_10_points_green_mean_change_c": round(
        float(effects["effect"].mean()), 3
    ),
    "notes": (
        "The trees learn the surface temperature minus air_temperature_share times the air temperature of the summer (air_temperature_column); "
        "the share is added back when predicting. landsat_cover is a measurement condition (the share of clear satellite pixels), not a cause: "
        "the what-if keeps it at its latest observed value. precip_clim and temp_clim are the climate normals of 2000-2012."
    ),
}
(MODELS / "model_card.json").write_text(json.dumps(model_card, indent=1, default=float))

loaded = load_final(model_card, MODELS)
assert np.allclose(
    loaded.predict(table[FEATURES_NEW]),
    final_model.predict(table[FEATURES_NEW]),
    atol=1e-9,
), "the saved model gives other predictions"
file = MODELS / model_card["model_file"]
print(
    f"saved: {file} ({file.stat().st_size / 1e6:.1f} MB), model_card.json and exam_predictions.csv"
)

**What we learned (Part 6)**
- **The exam (2023-2025, each year predicted by a model that has not seen it):** the final model has an RMSE of 1.46 / 1.32 / 1.01 °C in 2023 / 2024 / 2025 (**1.28 °C** over the three years), against 1.70 / 1.48 / 1.12 °C (1.45 °C) for the model before Part 5, 1.85 °C for Ridge and 1.91 °C for the district's own past mean. The R² of the city level is 0.912 (0.875 before), inside the cities 0.726 (0.710 before; Ridge 0.312).
- **The weather uncertainty costs 0.28 °C:** with the scenario weather (mean of the 5 summers before) the RMSE is 1.56 °C instead of 1.28 °C (before Part 5: 1.73 against 1.45); the level of the cities gets worse (R² 0.912 to 0.840), the differences inside the cities do not (0.726 to 0.734). The gain of Part 5 is the same with the scenario weather (-0.18 °C) as with the real weather (-0.18 °C).
- **A limit that stays:** inside the cities the district's *own past* explains the differences between districts slightly better (R² 0.754) than the model (0.726), but the gap has shrunk (0.710 before); the model is clearly better at the level of the cities (0.912 against 0.727) and, unlike the past mean, it also works for districts and cities without a history and it can answer the what-if.
- **The bias is smaller, but it is still there:** measured minus predicted is +0.50, +0.22 and +0.21 °C in the three years (before Part 5: +0.90, +0.56, +0.39 °C). The models still predict a little too cool because the recent summers were hotter; the air temperature offset follows the hotter summers better than trees alone. The what-if uses a *difference* of two predictions and does not carry this bias.
- **What-if:** +10 points of green share lowers the predicted surface temperature by **0.63 °C on average, in every district** (the monotone constraint). It is strongest in the Southeast (-0.70 °C) and weakest in Marmara (-0.47 °C). Before Part 5 it was -0.72 °C. The likely reason is that the new climate columns take over a part of the difference *between* cities that the green share had picked up (this is the reading of the numbers, not a separate test). The what-if effect (-0.63 °C) now sits closer to the observed association of the what-if notebook (-0.48 °C) than the -0.72 °C of the model before Part 5; the two numbers describe different things (what-if Part 4).
- **What the model uses:** the air temperature of the summer matters most (shuffling it raises the RMSE by 1.7 °C), then the green share (0.5 °C, second); then the longitude, the climate normal of the temperature and the elevation (0.2 °C), and the latitude, the clear sky, the hot days and the rain normal (0.1 °C). The within-city features count little because they explain only the small part inside the cities.
- **The final model** is trained on all years 2013-2025 and saved as one XGBoost file (15.7 MB) with its model card (which also holds the share of the air temperature) and the exam predictions in `data/models/`. Loading the file again gives the same predictions.